In [28]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
import operator


In [29]:
load_dotenv(find_dotenv())

True

In [30]:
generate_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
evaluate_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [31]:
## state 
class PostState(TypedDict):

    topic: str
    post : str
    evaluation : Literal['Approved', 'Improvement Needed']
    feedback : str
    iteration :int
    max_iteration : int
    

In [32]:
from pydantic import BaseModel, Field

class PostEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the Post.")
    

In [33]:
structured_evaluator_llm = evaluate_llm.with_structured_output(PostEvaluation)

In [34]:
def generate_post(state : PostState):

    messages = [
    SystemMessage(content="You are an insightful and engaging LinkedIn educator."),
    HumanMessage(content=f"""
Write a short, original, and thought-provoking LinkedIn post on the topic: "{state['topic']}".

Rules:
- Max 700 characters.
- Focus on clarity, inspiration, and professional growth.
- Share an educational insight, framework, or lesson learned.
- Use simple, accessible language with a professional tone.
- End with a reflective takeaway or call to action for readers.
""")
]
    
    ## send generator llm 

    response = generate_llm.invoke(messages).content 

    ## return response 
    return {'tweet': response }

In [35]:
def evaluate_post(state: PostState):

    # prompt
    messages = [
        SystemMessage(content="You are a rigorous LinkedIn content critic. You evaluate posts based on educational value, clarity, originality, and professional engagement."),
        HumanMessage(content=f"""
Evaluate the following LinkedIn post:

Post: "{state['post']}"

Use the criteria below to evaluate the post:

1. Originality – Does it bring a fresh perspective or recycled advice?  
2. Educational Value – Does it teach, explain, or inspire professional growth?  
3. Clarity – Is the language simple, structured, and easy to follow?  
4. Engagement Potential – Would professionals comment, share, or reflect on it?  
5. Format – Is it a well-formed LinkedIn post (under 700 characters, no excessive jargon, avoids clickbait)?

Auto-reject if:
- It exceeds 700 characters  
- It reads like vague motivational fluff without substance  
- It ends with generic throwaway lines (e.g., “Success is a journey” without context)  

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses  
""")
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [36]:
def optimize_post(state: PostState):

    messages = [
        SystemMessage(content="You refine LinkedIn posts for clarity, educational impact, and professional engagement based on given feedback."),
        HumanMessage(content=f"""
Improve the LinkedIn post based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Post:
{state['post']}

Re-write it as a concise, insightful LinkedIn post.  
Rules:
- Stay under 700 characters.  
- Focus on clarity, inspiration, and professional growth.  
- Share a practical insight, framework, or lesson learned.  
- Use simple, accessible language with a professional tone.  
- End with a reflective takeaway or call to action for readers.  
""")
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'post': response, 'iteration': iteration, 'post_history': [response]}

In [37]:
graph = StateGraph(PostState)

graph.add_node('generate', generate_post)
graph.add_node('evaluate', evaluate_post)
graph.add_node('optimize', optimize_post)

